In [1]:
import torch
import os
import numpy as np

import sys, os

import polars as pl

from cmm.topology import Topology
from cmm.settings import Settings, MolecularDynamicsSettings, PolarizationSettings, ShortRangeSettings
from cmm.system import System, create_system_from_ext_xyz_file, create_system_from_xyz_file
from cmm.force_fields.spcfw import SPCFW
from cmm.force_fields.cmm import CMM2
from cmm.units import BOHR2ANG, HARTREE2KCAL, HARTREE2KJ
from cmm.misc_utils import write_xyz

from cmm.drivers import SinglePointDriver, BatchSinglePointDriver, OptimizationDriver

In [2]:
data_path = "/home/heindelj/dev/julia_development/reactive_force_fields/notebooks/data"

settings = Settings()
settings.add_neighbor_list_settings(cutoff=30.0, padding=1.5)
settings.add_long_range_electrostatics_settings(use_long_range=False, cutoff=30.0)
settings.add_long_range_dispersion_settings(use_long_range=False)
settings.add("polarization", PolarizationSettings())
settings.add("short_range", ShortRangeSettings(cutoff=15.0))

w2_meili_systems = create_system_from_xyz_file(os.path.join(data_path, "w2_meili.xyz"), settings, requires_grad=True, device="cpu")
#w2_sobol_systems = create_system_from_xyz_file(os.path.join(data_path, "w2_sobol_dimers.xyz"), settings, requires_grad=False, device="cpu")
ref_cluster_systems = create_system_from_xyz_file("/home/heindelj/OneDrive/Documents/Coding_Projects/python_development/pyCMM/tests/data/water_clusters.xyz", settings, requires_grad=True, device="cpu")

w2_meili_energies = pl.read_csv(os.path.join(data_path, "w2_meili.csv"))

/home/heindelj/miniforge3/envs/pycmm/lib/python3.12/site-packages/torch/nested/__init__.py:226: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return _nested.nested_tensor(


In [3]:
bsp_driver = BatchSinglePointDriver("CMM")
bsp_driver.run(w2_meili_systems)

In [4]:
sp_driver = SinglePointDriver("CMM")
sp_driver.run(ref_cluster_systems[-1])
print(sp_driver.output['V_total'] * 627.51)

tensor(-258.7893, dtype=torch.float64, grad_fn=<MulBackward0>)


In [5]:
opt_driver = OptimizationDriver("CMM")
opt_driver.run(ref_cluster_systems[-1])
write_xyz("temp.xyz", opt_driver.system.labels, opt_driver.system.coords * BOHR2ANG)